# Exercise 2: Regression - portable notebook

This is the **portable version** of the Exercise 2 regression practice
from **Machine Learning for Neuroscience**, generated from the canonical
course notebook by `scripts/build_portable_notebook.py`. It is meant for
running or editing the code in Google Colab or in a local VS Code /
Jupyter setup.

The richer version -- with the feature-set comparison activity embedded
and running in the browser -- is the published course page:
<https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_02/exercise_02.html>

In this notebook the interactive activity is replaced by a link to that
page; every Python analysis cell is kept and runnable. Questions marked
*Think first* are followed, where one exists, by a collapsible *Check
your reasoning* block; open questions are left without one fixed answer.

## Setup

This notebook imports only `numpy`, `pandas`, `matplotlib` and
`scikit-learn`. All four are already installed on Google Colab, and in a
typical scientific-Python environment, so there is normally nothing to
do here.

If one of the imports further down fails, run the next cell once (edit
the version pins if your project needs specific ones), then restart the
kernel and run the notebook from the top. The notebook also downloads two
public data files the first time it runs, so it needs internet access.

In [ ]:
# If an import below fails, uncomment and run this line once, then
# restart the kernel. Safe on Colab, VS Code and Jupyter.
# %pip install numpy pandas matplotlib scikit-learn

# Exercise 2: Regression

## What this notebook covers

This is Exercise 2 of *Machine Learning for Neuroscience*. Exercise 2 is about
**regression and the bias-variance tradeoff**. This practice covers the first
part: **linear regression on real neuroimaging data**. A later practice adds
k-nearest neighbours and the bias-variance tradeoff itself.

In this part you will:

1. load a wide *modelling table* -- ABIDE-II phenotype columns joined to
   FreeSurfer brain measurements for the same participants;
2. build one honest linear-regression workflow: split, fit on training data,
   predict held-out data, score it;
3. compare that honest score with the *training* score and with a deliberately
   *invalid* fit-on-the-test-set score, to see what "evaluation" can and cannot
   mean;
4. use a browser activity to compare feature sets side by side;
5. see what happens to an unregularised fit when the feature count is
   comparable to or larger than the sample size, and whether ridge / lasso
   regression can help;
6. see what does -- and does not -- change as the sample gets smaller.

**Prerequisites:** the regression lecture; comfort with `pandas`, `numpy`,
`matplotlib`, and the scikit-learn `fit` / `predict` pattern.

## 1. The modelling table

Each **row is one participant**. The columns fall into three kinds:

- **phenotype columns** -- possible outcomes and context: `age`, `sex`,
  diagnostic `group`, and cognitive / behavioural scores such as `FIQ`
  (full-scale IQ) merged in from the ABIDE-II phenotypic file;
- **brain columns** -- one FreeSurfer measurement (`fsCT` cortical thickness,
  `fsArea` surface area, `fsVol` grey-matter volume, `fsLGI` gyrification) for
  one cortical region of one hemisphere, e.g. `fsCT_L_46_ROI`. The regions are
  the 360 parcels of the HCP-MMP1 (Glasser) atlas;
- **identifiers** -- `subject`, `site`.

This exercise's main outcome is **`age`**: it is a native column of the brain
table itself (no join, no missing values, all 1004 participants), unlike every
cognitive/behavioural score, which requires merging in the phenotypic file and
is missing for some participants. `FIQ` reappears in Section 5, where it is
used deliberately as a *harder* target.

This is a *modelling* table: wide, one measurement per column, ready for
`X` / `y`. It is not the small phenotype-only table from Exercise 1.

In [ ]:
# Data loading. In the published book this cell is collapsed; it is plain,
# runnable Python -- two public CSVs pinned to an immutable commit, merged on the
# participant id. Nothing here is repository-specific.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

PIN = "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b"
BASE = f"https://raw.githubusercontent.com/neurohackademy/nh2020-curriculum/{PIN}/tu-machine-learning-yarkoni/data"

brain = pd.read_csv(f"{BASE}/abide2.tsv", sep="\t")
brain = brain.loc[:, [c for c in brain.columns if not str(c).startswith("Unnamed")]]
phen = pd.read_csv(f"{BASE}/abide2_phenotypic.csv", encoding="latin-1", low_memory=False)
phen.columns = phen.columns.str.strip()

KEEP_PHEN = ["FIQ", "VIQ", "PIQ", "SRS_TOTAL_RAW", "ADOS_G_TOTAL", "ADI_R_SOCIAL_TOTAL_A"]
model_df = brain.merge(phen[["SUB_ID", *KEEP_PHEN]], left_on="subject", right_on="SUB_ID", how="left")
model_df = model_df.drop(columns=["SUB_ID"])

BRAIN_COLS = [c for c in model_df.columns if c.startswith("fs")]
PHENO_COLS = ["age", "age_resid", "sex", "group", *KEEP_PHEN]
print(f"modelling table: {model_df.shape[0]} participants x {model_df.shape[1]} columns")

In [ ]:
# A compact look: the phenotype columns plus three representative brain columns,
# then a separate count summary. (Printing all 1446 columns would tell you
# nothing.)
def measure_of(col):
    return col.split("_")[0]          # fsCT, fsArea, fsVol, fsLGI

def roi_of(col):
    return col.split("_", 2)[2].rsplit("_ROI", 1)[0]   # "L_46" etc.

preview_cols = ["subject", "site", "group", "age", "sex", "FIQ",
                "fsCT_L_46_ROI", "fsArea_L_IPS1_ROI", "fsVol_R_V1_ROI"]
display(model_df[preview_cols].head(4))

measures = sorted({measure_of(c) for c in BRAIN_COLS})
rois = sorted({roi_of(c) for c in BRAIN_COLS})
print(f"phenotype columns : {len(PHENO_COLS)}  -> {PHENO_COLS}")
print(f"brain features    : {len(BRAIN_COLS)}  = {len(measures)} measures x {len(rois)} region-hemispheres")
print(f"measurement types : {measures}")
print(f"missing brain cells: {int(model_df[BRAIN_COLS].isna().sum().sum())}")
print()
print("age available for", int(model_df['age'].notna().sum()), "of", len(model_df), "participants")
print("FIQ available for", int(model_df['FIQ'].notna().sum()), "of", len(model_df), "participants")

#### Think first

Before going on, name -- for this table -- (a) this exercise's main outcome,
(b) roughly how many columns could be *features*, (c) what a single row is,
(d) the four measurement types, and (e) how a brain column encodes both a
region and a hemisphere.

<details>
<summary><strong>Check your reasoning</strong></summary>

Outcome: `age` (in years; `FIQ` reappears later as a harder target). Features:
the ~1440 `fs...` brain columns. A row: one participant. Measurement types:
`fsCT`, `fsArea`, `fsVol`, `fsLGI`. A brain column name is
`fs<measure>_<hemisphere>_<region>_ROI`, so `fsCT_L_46_ROI` is cortical
thickness of left area 46.

</details>

## 2. One honest linear-regression workflow

**Target:** `age`, in years. **Features:** every eligible cortical-thickness
(`fsCT`) region, bilateral -- 358 features. Age-related cortical thinning is a
whole-cortex phenomenon (Bethlehem et al. 2022; Storsve et al. 2014), not one a
small hand-picked region subset would represent fairly, so this recipe uses the
full eligible region set for one measurement type rather than a literature
bundle. 358 features stays well under the ~750 training rows below, so ordinary
least squares is well posed (Section 5 pushes past that boundary on purpose).

The workflow, step by step:

In [ ]:
# The full set of eligible cortical-thickness columns: every bilateral HCP-MMP1
# region (this excludes "5L"/"5R", the two regions that exist in only one
# hemisphere and so cannot form a bilateral pair).
ASYMMETRIC_ROIS = ("_5L_ROI", "_5R_ROI")
FEATURES = [c for c in BRAIN_COLS if c.startswith("fsCT_") and not c.endswith(ASYMMETRIC_ROIS)]
assert all(c.startswith("fsCT_") for c in FEATURES)           # brain-only
assert "age" not in FEATURES and "FIQ" not in FEATURES        # no target / phenotype leakage
print(f"{len(FEATURES)} features, e.g. {FEATURES[:3]}")

In [ ]:
# 1-2. brain-only X and target y; age has no missing values, but the same
#      drop-missing-target step from Exercise 2's other analyses is kept here
#      for consistency.
has_age = model_df["age"].notna()
X = model_df.loc[has_age, FEATURES].to_numpy(float)
y = model_df.loc[has_age, "age"].to_numpy(float)
groups = model_df.loc[has_age, "group"].to_numpy()      # 1 = autism, 2 = control

# 3. one fixed, reproducible split. Stratify by diagnosis so train and test have
#    a similar autism / control mix -- without turning this into a splitting
#    lecture.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=groups
)
print(f"n_train = {len(y_train)}   n_test = {len(y_test)}   n_features = {X.shape[1]}")

In [ ]:
# 4-7. scale + fit on the TRAINING rows only (Pipeline keeps the scaler honest);
#      predict the untouched test rows; score.
model = make_pipeline(StandardScaler(), LinearRegression())
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
test_r2 = r2_score(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
print(f"held-out R^2 = {test_r2:.3f}")
print(f"held-out MSE = {test_mse:.1f}  (RMSE {test_mse**0.5:.1f} years)")

In [ ]:
# 8. observed vs predicted on the test set, with the perfect-prediction diagonal.
fig, ax = plt.subplots(figsize=(4.4, 4.4))
lims = [min(y_test.min(), y_pred.min()) - 3, max(y_test.max(), y_pred.max()) + 3]
ax.plot(lims, lims, "--", color="0.4", lw=1, label="perfect prediction")
ax.scatter(y_test, y_pred, s=16, alpha=0.5)
ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
ax.set_xlabel("observed age (years)"); ax.set_ylabel("predicted age (years, held-out)")
ax.set_title(f"held-out R$^2$ = {test_r2:.3f}")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

A few things to hold onto:

- the fitted model is a hyperplane in 358-dimensional feature space -- there is
  nothing useful to *draw* there. The observed-vs-predicted plot is the honest
  picture: points hugging the diagonal mean the model predicted well;
- this is a genuinely **positive held-out R²**: cortical thickness carries real
  information about age in this sample, unlike `FIQ` (Section 5). A positive
  result is not automatically a *large* one -- read the number, not just its
  sign;
- this split estimates generalisation to **new participants from the same 17
  ABIDE-II sites**. A random participant split says nothing about how the model
  would do on an entirely new scanner or site.

#### Think first

The test set here is a random 25% of participants, drawn from the *same* 17
sites, scanners, and protocols as the training set. Name one prediction task
this held-out score does **not** speak to.

<details>
<summary><strong>Check your reasoning</strong></summary>

It does not estimate performance on a **new site or scanner** the model has
never seen. Sites differ in scanner hardware, sequences, and the people they
recruit; a model tuned on these 17 can lean on site-linked quirks that will not
transfer. A leave-one-site-out split would be needed to speak to that.

</details>

## 3. Three ways to score the same model

Same fixed split, same 358-feature recipe. Three numbers:

| | fit on | evaluate on | what it estimates |
|---|---|---|---|
| **A. Correct** | training rows | untouched test rows | performance on new participants |
| **B. Training score** | training rows | those same training rows | how well it fits data it has seen (a diagnostic, not a generalisation estimate) |
| **C. Invalid** | **test rows** | those same test rows | nothing usable -- the data used to fit cannot also give an honest score |

#### Think first

Predict the order of the three R² values from largest to smallest, and say why.

In [ ]:
# A: correct (already computed above) -> reuse `model`, `y_test`, `y_pred`.
# B: resubstitution
train_r2 = r2_score(y_train, model.predict(X_train))
train_mse = mean_squared_error(y_train, model.predict(X_train))

# C: a DELIBERATELY INVALID model -- fit on the test rows, scored on the same
# test rows. Named so it cannot be reused by accident. Note n_test (251) is
# SMALLER than n_features (358) here, so this fit is not just optimistic --
# it is underdetermined, and can match the 251 test points almost exactly.
invalid_test_fitted_model = make_pipeline(StandardScaler(), LinearRegression())
invalid_test_fitted_model.fit(X_test, y_test)
invalid_pred = invalid_test_fitted_model.predict(X_test)
invalid_r2 = r2_score(y_test, invalid_pred)
invalid_mse = mean_squared_error(y_test, invalid_pred)

scores = pd.DataFrame(
    {
        "fit on": ["training rows", "training rows", "test rows (INVALID)"],
        "evaluated on": ["test rows", "training rows", "same test rows"],
        "R^2": [test_r2, train_r2, invalid_r2],
        "MSE": [test_mse, train_mse, invalid_mse],
    },
    index=["A. correct", "B. training score", "C. invalid"],
)
print(f"n_train = {len(y_train)}   n_test = {len(y_test)}   n_features = {X.shape[1]}")
scores.round(3)

In [ ]:
panels = [
    ("A. correct\n(fit train, test test)", y_test, y_pred, test_r2),
    ("B. training score\n(fit train, score train)", y_train, model.predict(X_train), train_r2),
    ("C. invalid\n(fit test, score test)", y_test, invalid_pred, invalid_r2),
]
allv = np.concatenate([y_train, y_test, y_pred, model.predict(X_train), invalid_pred])
lims = [allv.min() - 3, allv.max() + 3]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.9), sharex=True, sharey=True)
for ax, (title, obs, pred, r2) in zip(axes, panels):
    ax.plot(lims, lims, "--", color="0.4", lw=1)
    ax.scatter(obs, pred, s=10, alpha=0.4)
    ax.set_xlim(lims); ax.set_ylim(lims); ax.set_aspect("equal")
    ax.set_title(f"{title}\nR$^2$ = {r2:.3f}", fontsize=9)
    ax.set_xlabel("observed age")
axes[0].set_ylabel("predicted age")
plt.tight_layout(); plt.show()

- **B is not a competing model.** A high training R² next to a lower test R² is
  the signature of a model fitting some noise it cannot reproduce on new data.
- **C is not a competing model either.** With 358 features and only 251 test
  rows, `invalid_test_fitted_model` does not just look good -- it is
  **underdetermined** and reproduces those 251 points almost exactly, the same
  instability Section 5 discusses on purpose. Its R² is close to a perfect
  score by construction, not because it learned anything that generalises.
- B evaluates the model on its training data, whereas A and C are evaluated on
  the test data. A is the only valid estimate of performance on unseen
  participants; C is invalid because the test data were used for fitting.

<details>
<summary><strong>Check your reasoning: the ordering</strong></summary>

C (invalid) > B (training) > A (correct). C is highest -- here essentially a
perfect fit -- because the same 251 rows picked the coefficients *and* graded
them, and with more features (358) than rows (251) the fit can match those
particular points almost exactly. B is next: the model saw these 753 rows while
fitting, so it reproduces them better than genuinely new rows. A is lowest
because the test rows had no influence on the coefficients at all -- it is the
only number that estimates anything about participants the model has not seen.

</details>

## 4. Comparing feature sets

Cortical structure changes with age across most of the brain, not in one
circumscribed system: large multi-site lifespan mapping (Bethlehem et al.,
2022,
[doi:10.1038/s41586-022-04554-y](https://doi.org/10.1038/s41586-022-04554-y))
and longitudinal work on adult cortical change (Storsve et al., 2014,
[doi:10.1523/JNEUROSCI.0391-14.2014](https://doi.org/10.1523/JNEUROSCI.0391-14.2014))
both describe widespread, region-varying change with age rather than change
confined to a small set of regions. So, unlike an IQ-specific theory that
predicts *where* to look first (Section 5), there is no single small region
set this exercise's literature prefers for age. The bundles below -- including
one originally assembled from IQ literature -- are used here simply as several
differently sized, differently located anatomical comparisons.

The activity compares two models at a time on **one fixed cohort and one fixed
set of folds**, so any difference is a real difference between feature sets. It
shows out-of-sample scores only -- never training scores.

### Compare feature sets on the course website

The interactive activity lets you configure two linear-regression models
-- a measurement type and an anatomical ROI bundle each -- and compares
their cross-validated performance on one fixed cohort and one fixed set
of folds.

> **Interactive version on the course website.** It is embedded in the
> published Exercise II page:
> <https://yoavmp.github.io/ml-neuro-tutorials/chapters/chapter_02/exercise_02.html>
> This portable notebook links to it instead of embedding it. The
> Python sections below still run the same kind of comparison directly.

#### Think first

1. Hold the ROI bundle fixed and change only the measurement type; then hold the
   measurement fixed and change only the bundle. Which control moves R² more?
2. Set one model to `All eligible ROIs`. What happens to the feature count and
   to R², and why -- and is that the same direction of change you would expect
   for a barely-predictable target like `FIQ` (Section 5)?
3. Find the single best-scoring configuration you can. Would you trust its R²
   as your final reported number? Why or why not?

Trying many configurations is **exploratory model comparison**. If you pick the
best-looking one and quote its cross-validated R² as "the" performance, that
number is optimistic -- you chose it *because* it looked good on this data. A
locked test set or a nested procedure is needed for an honest final claim.

## 5. Regularisation preview: age vs FIQ with the full brain

So far, `age` has used one measurement type (358 features). What happens with
**all four measurement types together** -- 1432 brain features -- and, for
comparison, the same recipe applied to **`FIQ`** (Exercise 2's harder target)?

`FIQ` is here specifically because intelligence has its own regional
hypothesis -- the Parieto-Frontal Integration Theory (Jung & Haier, 2007,
[doi:10.1017/S0140525X07001185](https://doi.org/10.1017/S0140525X07001185))
and cortical-thickness-and-IQ work (Narr et al., 2007,
[doi:10.1093/cercor/bhl125](https://doi.org/10.1093/cercor/bhl125)) motivated
the `Frontoparietal` bundle offered in Section 4. That literature motivates a
*hypothesis about where to look*, not a guaranteed prediction: no
linear-regression feature set in this exercise gives `FIQ` a positive held-out
score (Section 4), and this section checks whether regularisation over the
*entire* brain changes that conclusion.

When the number of features `p` is comparable to, or larger than, the number of
training participants `n`, unregularised linear regression can be **unstable
or underdetermined**: with `p >= n_train`, ordinary least squares does not have
a unique solution (scikit-learn returns a minimum-norm solution instead), and
the fit can chase noise the way Section 3's invalid model did. **Ridge**
regression shrinks coefficients toward zero but normally keeps every feature;
**lasso** regression can shrink some coefficients *exactly* to zero, which is a
crude form of feature selection. Neither is "marginally reducing" the feature
count the way choosing a smaller anatomical bundle does -- ridge does not
reduce it at all, and lasso's zeroed coefficients are a side effect of the
penalty, not a reviewed anatomical choice. Proper feature selection and
regularisation are covered in a later lesson; this is a preview, not that
lesson.

This section fits `Pipeline(StandardScaler(), RidgeCV(...))` and
`Pipeline(StandardScaler(), LassoCV(...))` over documented logarithmic alpha
grids, with `RidgeCV`/`LassoCV` choosing `alpha` from a 5-fold cross-validation
**on the training rows only**; the held-out test set is touched exactly once,
after the model is fully fixed. A reproducible nested 5x5 cross-validation
audit (`scripts/regression_model_audit.py`, `scripts/regression_model_audit_result.json`)
confirms the same conclusion with every target/feature-space/model combination
evaluated the same isolated way -- see the WP12 report for the full table.

In [ ]:
# The full brain: all four measures, every bilateral region -- 1432 features.
import warnings
from sklearn.exceptions import ConvergenceWarning

MEASURE_PREFIXES = ["fsCT", "fsArea", "fsVol", "fsLGI"]
ALL_MEASURES_FEATURES = [
    c for c in BRAIN_COLS
    if c.split("_")[0] in MEASURE_PREFIXES and not c.endswith(ASYMMETRIC_ROIS)
]
print(f"{len(ALL_MEASURES_FEATURES)} features across {len(MEASURE_PREFIXES)} measures")

RIDGE_ALPHAS = np.logspace(-1, 6, 29)     # documented logarithmic grid
LASSO_ALPHAS = np.logspace(-3, 2, 26)     # documented logarithmic grid
INNER_CV = KFold(n_splits=5, shuffle=True, random_state=1)

def fit_and_score(target, features=ALL_MEASURES_FEATURES, seed=42):
    """One fixed holdout split (same protocol as Section 2); OLS, ridge and
    lasso, with ridge/lasso choosing alpha from INNER_CV on the training rows
    only. The held-out split is evaluated once per model, never used to pick
    the target, the feature set, the model family, or alpha."""
    m = model_df[target].notna()
    Xt = model_df.loc[m, features].to_numpy(float)
    yt = model_df.loc[m, target].to_numpy(float)
    g = model_df.loc[m, "group"].to_numpy()
    Xtr, Xte, ytr, yte = train_test_split(Xt, yt, test_size=0.25, random_state=seed, stratify=g)

    rows = []
    ols = make_pipeline(StandardScaler(), LinearRegression()).fit(Xtr, ytr)
    p, n_tr = Xtr.shape[1], Xtr.shape[0]
    ols_pred = ols.predict(Xte)
    rows.append({"target": target, "model": "OLS (unregularised)",
                 "R^2": r2_score(yte, ols_pred), "MSE": mean_squared_error(yte, ols_pred),
                 "alpha": np.nan, "non-zero coefs": p,
                 "note": f"underdetermined: p={p} >= n_train={n_tr}" if p >= n_tr else ""})

    ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS, cv=INNER_CV)).fit(Xtr, ytr)
    ridge_pred = ridge.predict(Xte)
    ridge_alpha = ridge.named_steps["ridgecv"].alpha_
    rows.append({"target": target, "model": "Ridge", "R^2": r2_score(yte, ridge_pred),
                 "MSE": mean_squared_error(yte, ridge_pred), "alpha": ridge_alpha,
                 "non-zero coefs": p, "note": ""})

    lasso_cv = LassoCV(alphas=LASSO_ALPHAS, cv=INNER_CV, max_iter=50000, tol=1e-3, n_jobs=-1, random_state=1)
    lasso = make_pipeline(StandardScaler(), lasso_cv)
    with warnings.catch_warnings():
        # The smallest alphas in a 26-point logarithmic grid can be slow to
        # converge to tol=1e-3 well within max_iter; the CV-selected alpha
        # (printed below) always converges cleanly, so this only silences
        # warnings from alphas LassoCV evaluates and does not end up choosing.
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        lasso.fit(Xtr, ytr)
    lasso_pred = lasso.predict(Xte)
    lasso_alpha = lasso.named_steps["lassocv"].alpha_
    nnz = int(np.sum(np.abs(lasso.named_steps["lassocv"].coef_) > 1e-10))
    rows.append({"target": target, "model": "Lasso", "R^2": r2_score(yte, lasso_pred),
                 "MSE": mean_squared_error(yte, lasso_pred), "alpha": lasso_alpha,
                 "non-zero coefs": nnz, "note": ""})
    return rows, n_tr, len(yte)

results = []
for target in ("age", "FIQ"):
    rows, n_tr, n_te = fit_and_score(target)
    results.extend(rows)
    print(f"{target}: n_train={n_tr}  n_test={n_te}  p={len(ALL_MEASURES_FEATURES)}")

preview = pd.DataFrame(results).set_index(["target", "model"])
preview[["R^2", "MSE", "alpha", "non-zero coefs"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
labels = [f"{t}\n{m}" for t, m in preview.index]
colors = ["0.65" if "OLS" in m else ("#2a6f9e" if t == "age" else "#b5622f")
          for t, m in preview.index]
ax.bar(range(len(preview)), preview["R^2"].to_numpy(), color=colors)
ax.axhline(0, color="0.3", lw=1)
ax.set_xticks(range(len(preview)), labels, fontsize=8)
ax.set_ylabel("held-out R$^2$ (p = 1432)")
ax.set_title("Grey = unregularised OLS (underdetermined here); colour = ridge / lasso")
plt.tight_layout(); plt.show()

Two different, both honest, conclusions from the same procedure:

- **`age`**: unregularised OLS on all 1432 features is *worse* than the
  358-feature recipe from Section 2 -- it is underdetermined and the fit
  chases noise. Ridge and lasso both **fix that instability and clearly beat**
  the 358-feature OLS result, using information from all four measurement
  types. Regularisation helped because there was a real signal to recover.
- **`FIQ`**: ununregularised OLS collapses to a deeply negative score, exactly
  as unstable as `age`'s. Ridge and lasso rescue it from that collapse -- but
  only back to **approximately zero**, not to a positive score. Lasso keeps a
  small number of non-zero coefficients out of 1432 and still finds nothing
  reliable. **Regularisation stabilised an unstable fit; it did not manufacture
  a signal that is not in the data.** Reported honestly, not cherry-picked: no
  split, alpha, or feature subset was chosen after looking at this result (see
  the WP12 report's full audit table for every configuration tested).

The two targets' MSE values are **not comparable** to each other -- age is
measured in years, FIQ in IQ-score points, different scales entirely. R² is the
only number in this table that means the same thing for both.

## 6. What does sample size change?

The cleanest test: keep `age`, keep the 358-feature recipe, keep one fixed
held-out set from Section 2, and vary the training size. Resample many times at
each size so the answer is about the *average* and its spread, not one lucky or
unlucky draw. (Comparing two different outcomes, such as `age` and `FIQ`,
cannot isolate a sample-size effect on its own -- they differ in scale,
reliability, and how they were measured. Section 5 already showed the honest
`age`-vs-`FIQ` comparison that matters here; this section holds the target
fixed and only sample size varies.)

In [ ]:
rng = np.random.default_rng(0)
sizes = [370, 470, 570, 670, len(y_train)]   # smallest size stays >= n_features (358)
n_rep = 40
lc_mean, lc_lo, lc_hi = [], [], []
for n in sizes:
    reps = []
    for _ in range(n_rep):
        idx = rng.choice(len(y_train), size=min(n, len(y_train)), replace=False)
        pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train[idx], y_train[idx])
        reps.append(r2_score(y_test, pipe.predict(X_test)))
    reps = np.array(reps)
    lc_mean.append(reps.mean()); lc_lo.append(np.percentile(reps, 10)); lc_hi.append(np.percentile(reps, 90))

print(f"n_features (p) = {X.shape[1]};  smallest training size {sizes[0]}  -> n/p = {sizes[0]/X.shape[1]:.2f}")
for n, m, lo, hi in zip(sizes, lc_mean, lc_lo, lc_hi):
    print(f"  n_train = {n:4d}   held-out R^2  mean {m:+.3f}   10-90 pct [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.fill_between(sizes, lc_lo, lc_hi, alpha=0.25, label="10th-90th percentile")
ax.plot(sizes, lc_mean, "o-", label="mean held-out R$^2$")
ax.axhline(0, color="0.5", lw=1, ls=":")
ax.set_xlabel("training-set size"); ax.set_ylabel("held-out R$^2$")
ax.set_title("age learning curve (all-eligible cortical thickness, p = 358)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

#### What the learning curve shows -- and does not

The smallest size here (370) is deliberately just above the feature count
(358, n/p ~= 1.03) -- barely enough rows to fit 358 coefficients at all, and the
spread of held-out R² is correspondingly wide, sometimes dipping below zero on
an unlucky draw. As the training set grows toward the full ~753 rows, the
spread collapses and the mean settles just under the Section 2 value. More data
bought **certainty about a real signal that was already there**, not a signal
the features did not carry -- contrast this with `FIQ` in Section 5, where more
regularisation could not manufacture a signal that was never there. Individual
random draws still wobble; the curve is about the average and its uncertainty.

## In summary

- An honest performance estimate fits on training data and scores on data the
  model has never seen. Scoring on the fitting rows -- whether they are the
  training set (optimistic) or, worse, the test set (invalid, and here
  literally underdetermined) -- inflates the number.
- `age` has a genuinely positive held-out R² from cortical structure alone;
  `FIQ` does not, with or without regularisation.
- When `p` is comparable to or larger than `n_train`, unregularised linear
  regression is unstable or underdetermined. Ridge and lasso can rescue that
  instability -- but only up to whatever real signal is in the data. For `age`
  that recovered a clearly better score; for `FIQ` it recovered only
  approximately zero, and that is reported as the honest result, not
  papered over.
- Comparing two different outcomes cannot isolate a sample-size effect; holding
  one target fixed and varying training size can. More data mainly bought
  *certainty*, both in the learning curve and in how confidently this exercise
  can say `age` is predictable from cortical structure and `FIQ`, here, is not.

Next practice: k-nearest neighbours, and the bias-variance tradeoff.

### Questions to take away

1. For this table, which columns are the outcome and which are the features?
2. Why does fitting and scoring a model on the *same* observations inflate its
   apparent performance? In Section 3, why is `invalid_test_fitted_model`'s
   score close to a perfect score rather than just "somewhat inflated"?
3. Section 2 gets a *positive* held-out R² for `age`. What would a *negative*
   one have meant instead?
4. Two feature sets in Section 4 give different cross-validated R². List three
   reasons other than "one set is biologically better" that could explain the
   gap.
5. In Section 5, ridge substantially improves `age`'s score but only brings
   `FIQ` back to approximately zero. What does that difference tell you about
   what regularisation can and cannot do?
6. Why can't comparing `age` and `FIQ`'s held-out scores, by itself, isolate a
   sample-size effect the way Section 6's learning curve can?
7. In the learning curve, what improves as the training set grows, and what
   does not?